# 自定义args_schema


In [ ]:
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from email.policy import default

from langchain_community.chat_models import ChatZhipuAI
from dotenv import load_dotenv
import os
from rich import print as rprint
from langchain_core.utils.function_calling import convert_to_openai_tool
from pydantic import BaseModel, Field
from scripts.regsetup import description

# 自定义args_schema
######1、提供大模型#########
ZHIPUAI_API_KEY = os.getenv("ZHIPUAI_API_KEY")
ZHIPUAI_API_URL = os.getenv("ZHIPUAI_BASE_URL")

load_dotenv(override=True)
model = ChatZhipuAI(
    model="glm-4-plus",
    # api_key=ZHIPUAI_API_KEY,
    # base_url=ZHIPUAI_API_URL,
)


class weatherSchema(BaseModel):
    city: str = Field(default="杭州", description="城市名称")
    if_forecast: bool = Field(default=False, description="是否包含明日天气预报")


@tool("get_weather", description="查询当日天气和明天", args_schema=weatherSchema)
def get_weather(city: str, if_forecast: bool):
    res = f"{city}今天天气不错"
    if if_forecast:
        res += "\n明天也不错"
    return res


rprint(convert_to_openai_tool(weatherSchema))

# 只有一个工具
model_with_tools = model.bind_tools([weatherSchema])

messages = [HumanMessage(content="今天北京天气如何？明天呢?")]
response = model_with_tools.invoke(messages)
messages.append(response)
tool_calls = response.tool_calls

for tool_call in tool_calls:
    if tool_call["name"] == "get_weather":
        tool_msg = get_weather.invoke(tool_call)
        messages.append(tool_msg)

final_response = model.invoke(messages)
messages.append(final_response)
for msg in messages:
    msg.pretty_print()


# 多个工具调用

In [ ]:
from langchain.tools import tool
from langchain.messages import HumanMessage


@tool
def get_weather(city: str) -> str:
    """
    获取当日天气
    Args:
    city: 城市名称
    """
    return f'{city}当天晴朗'


@tool("io")
def get_news() -> str:
    """
    获取当日新闻
    """
    return "近期，受全球储蓄芯片短缺等多重因素影响，多地回收商称废旧手机回收市场迎来“火热潮”，回收价格普遍上涨，旧手机成“香饽饽”。"


model_with_tools = model.bind_tools([get_weather, get_news])
messages = [
    HumanMessage("今天深圳天气如何？今天新闻是什么？你是谁？")
]

response = model_with_tools.invoke(messages)
response.pretty_print()
messages.append(response)
for tool_call in response.tool_calls:
    if tool_call["name"] == "get_weather":
        tool_msg = get_weather.invoke(tool_call)
        print(tool_msg)
        messages.append(tool_msg)
    elif tool_call["name"] == "get_news":
        tool_msg = get_news.invoke(tool_call)
        print(tool_msg)
        messages.append(tool_msg)
    else:
        raise Exception("不存在的工具")
        final_response = model.invoke(messages)
        messages.append(final_response)
#让ai回答
finnal_response = model.invoke(messages)
messages.append(finnal_response)

for msg in messages:
    msg.pretty_print()

# 三个工具并且调用

In [9]:
from langchain.tools import tool
from rich import print as rprint
from langchain.messages import HumanMessage
from langchain_community.chat_models import ChatZhipuAI
import os
from dotenv import load_dotenv
from langchain.tools import tool
#定义工
# 用args的Goole风格
@tool
def get_color(color: str="红色", if_forecast: bool=False):
    """获取颜色

        Args:
            color:颜色名称
            if_forecast:是否还有别的颜色
    """
    default_color = "red"
    res = f"{color}颜色很好"
    if if_forecast:
        res += "\n没见过这个颜色"
    return res
# rprint(convert_to_openai_tool(get_color))

# 定云朵
@tool
def get_cloud(cloud: str="白色", if_forecast: bool=False):
    """
    获取云朵的颜色

    Args:
        cloud: 云的颜色
        if_forecast: 是否还有别的云的颜色
    """
    res = f"{cloud}的云很美"
    if if_forecast:
        res += "\n有兴趣我去了解"
    return res
# rprint(convert_to_openai_tool(get_cloud))

# 定天气
@tool
def get_city(city: str, if_forecast: bool=False):
    """
    获取城市的天气

    Args:
        city: 城市名称
        if_forecast: 是否还有别的城市情况
    """
    res = f"{city}的天气不错"
    if if_forecast:
        res += "\n可能会下雨"
    return res
#rprint(convert_to_openai_tool(get_city))

# ============================================
# 绑定工具
# ============================================
load_dotenv(override=True)
model = ChatZhipuAI(
   model="glm-4.5-Air",

)
model_with_tools = model.bind_tools([get_city, get_cloud, get_color])
messages = [
    HumanMessage(   "请使用 get_city、get_cloud、get_color 这三个工具，"
        "分别获取：厦门的天气、云朵颜色、你喜欢的颜色,为什么会喜欢这个颜色？")
]

response = model_with_tools.invoke(messages)
response.pretty_print()
messages.append(response)
for tool_call in response.tool_calls:
    if tool_call["name"] == "get_city":
        tool_msg = get_city.invoke(tool_call)
        print(tool_msg)
        messages.append(tool_msg)
    elif tool_call["name"] == "get_cloud":
        tool_msg = get_cloud.invoke(tool_call)
        print(tool_msg)
        messages.append(tool_msg)
    elif tool_call["name"] == "get_color":
        tool_msg = get_color.invoke(tool_call)
        print(tool_msg)
        messages.append(tool_msg)
    else:
        raise Exception("不存在的工具")
        final_response = model.invoke(messages)
        messages.append(final_response)
#让ai回答
finnal_response = model.invoke(messages)
messages.append(finnal_response)

for msg in messages:
    msg.pretty_print()

================================== Ai Message ==================================


我来为您调用这三个工具来获取相关信息。
Tool Calls:
  get_city (call_-7436633687162086256)
 Call ID: call_-7436633687162086256
  Args:
    city: 厦门
  get_cloud (call_-7436633687162086255)
 Call ID: call_-7436633687162086255
  Args:
  get_color (call_-7436633687162086254)
 Call ID: call_-7436633687162086254
  Args:
content='厦门的天气不错' name='get_city' tool_call_id='call_-7436633687162086256'
content='白色的云很美' name='get_cloud' tool_call_id='call_-7436633687162086255'
content='红色颜色很好' name='get_color' tool_call_id='call_-7436633687162086254'
================================ Human Message =================================

请使用 get_city、get_cloud、get_color 这三个工具，分别获取：厦门的天气、云朵颜色、你喜欢的颜色,为什么会喜欢这个颜色？
================================== Ai Message ==================================


我来为您调用这三个工具来获取相关信息。
Tool Calls:
  get_city (call_-7436633687162086256)
 Call ID: call_-7436633687162086256
  Args:
    city: 厦门
  get_cloud (call_-7436633687